In [1]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load Titanic dataset directly
df = sns.load_dataset("titanic")

# Select required columns
df = df[['survived', 'pclass', 'sex', 'age', 'fare']]

# Remove missing values
df['age'] = df['age'].fillna(df['age'].median())

# Convert sex into numbers
df['sex'] = LabelEncoder().fit_transform(df['sex'])

# Input and output
X = df.drop('survived', axis=1)
y = df['survived']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------- K-FOLD ----------------

kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = DecisionTreeClassifier(random_state=42)

scores = []

for train_index, val_index in kf.split(X_train):

    X_train_fold = X_train.iloc[train_index]
    X_val_fold = X_train.iloc[val_index]

    y_train_fold = y_train.iloc[train_index]
    y_val_fold = y_train.iloc[val_index]

    model.fit(X_train_fold, y_train_fold)

    y_pred = model.predict(X_val_fold)

    accuracy = accuracy_score(y_val_fold, y_pred)

    scores.append(accuracy)

print("K-Fold Accuracy Scores:")
print(scores)

print("Mean K-Fold Accuracy:")
print(sum(scores) / len(scores))


# ---------------- GRID SEARCH ----------------

parameters = {
    'max_depth': [2, 3, 4, 5, 6, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    parameters,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest Grid Search Accuracy:")
print(grid.best_score_)


# ---------------- TEST BEST MODEL ----------------

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\nTest Accuracy:")
print(accuracy_score(y_test, y_pred))

K-Fold Accuracy Scores:
[0.7902097902097902, 0.7762237762237763, 0.7535211267605634, 0.7746478873239436, 0.7535211267605634]
Mean K-Fold Accuracy:
0.7696247414557273

Best Parameters:
{'criterion': 'entropy', 'max_depth': 3, 'min_samples_split': 2}

Best Grid Search Accuracy:
0.8145769723234512

Test Accuracy:
0.7988826815642458
